# Codificação de Fonte — Huffman e LZ78

Grupo 04.

Duas codificações sobre o mesmo texto. Cada uma gera um bitstream que o professor corrompe com erros simulando um canal ruidoso. Depois decodificamos os arquivos corrompidos e comparamos o impacto.

In [ ]:
import heapq, json, os
from collections import Counter
from dataclasses import dataclass, field
from typing import Any

ARQUIVO_TEXTO = "texto_original.txt"

for d in ("enviar", "recebido", "decodificado"):
    os.makedirs(d, exist_ok=True)

with open(ARQUIVO_TEXTO, "r", encoding="utf-8") as f:
    texto_original = f.read()


def bits_to_bytes(bitstream):
    pad = (8 - len(bitstream) % 8) % 8
    padded = bitstream + "0" * pad
    return bytes(int(padded[i:i+8], 2) for i in range(0, len(padded), 8))


def bytes_to_bits(data, bit_length=None):
    bits = "".join(format(b, '08b') for b in data)
    return bits[:bit_length] if bit_length is not None else bits


tamanho_original = os.path.getsize(ARQUIVO_TEXTO)
print(f"{len(texto_original)} chars, {tamanho_original} bytes")
print(texto_original)

## Huffman

Códigos de comprimento variável. Cada caractere recebe um código com tamanho inversamente proporcional à sua frequência.

Frequência e probabilidade dos caracteres no texto.

In [ ]:
freq = Counter(texto_original)
total = len(texto_original)

for ch, f in sorted(freq.items(), key=lambda x: -x[1]):
    print(f"{repr(ch):<6} {f:>4}  {f/total:.4f}")

print(f"\n{len(freq)} símbolos distintos")

Construção da árvore de Huffman e geração do codebook.

In [ ]:
@dataclass(order=True)
class HuffmanNode:
    freq: int
    char: Any = field(compare=False, default=None)
    left: Any = field(compare=False, default=None)
    right: Any = field(compare=False, default=None)


def build_huffman_tree(text):
    heap = [HuffmanNode(f, ch) for ch, f in Counter(text).items()]
    heapq.heapify(heap)

    if len(heap) == 1:
        return HuffmanNode(freq=heap[0].freq, left=heapq.heappop(heap))

    while len(heap) > 1:
        a = heapq.heappop(heap)
        b = heapq.heappop(heap)
        heapq.heappush(heap, HuffmanNode(freq=a.freq + b.freq, left=a, right=b))

    return heap[0]


def build_codes(node, prefix="", codebook=None):
    if codebook is None:
        codebook = {}
    if node.char is not None:
        codebook[node.char] = prefix or "0"
        return codebook
    if node.left:
        build_codes(node.left, prefix + "0", codebook)
    if node.right:
        build_codes(node.right, prefix + "1", codebook)
    return codebook


tree = build_huffman_tree(texto_original)
codebook_huffman = build_codes(tree)

for ch, code in sorted(codebook_huffman.items(), key=lambda x: len(x[1])):
    print(f"{repr(ch):<6} {code}")

Codificação do texto, empacotamento em bytes e gravação dos arquivos.

In [ ]:
def huffman_encode(text, codebook):
    return "".join(codebook[ch] for ch in text)


bitstream_huffman = huffman_encode(texto_original, codebook_huffman)
data_huffman = bits_to_bytes(bitstream_huffman)

with open("enviar/huffman_codificado.bin", "wb") as f:
    f.write(data_huffman)

with open("enviar/huffman_codebook.json", "w") as f:
    json.dump({"codebook": codebook_huffman, "bit_length": len(bitstream_huffman)},
              f, ensure_ascii=False, indent=2)

print(f"Original: {tamanho_original} bytes")
print(f"Huffman:  {len(data_huffman)} bytes ({len(data_huffman)/tamanho_original:.1%})")
print(f"          {len(bitstream_huffman)} bits úteis + {len(data_huffman)*8 - len(bitstream_huffman)} de padding")

## LZ78

Dicionário dinâmico. Tokens fixos: 16 bits de índice + 8 bits de caractere = 24 bits. Dicionário cresce até 65.535 entradas.

Geração dos tokens LZ78 a partir do texto.

In [ ]:
def lz78_encode(text):
    dictionary = {}
    tokens = []
    current = ""
    next_index = 1

    for ch in text:
        current += ch
        if current not in dictionary:
            prefix = current[:-1]
            tokens.append((dictionary.get(prefix, 0), ch))
            if next_index <= 65535:
                dictionary[current] = next_index
                next_index += 1
            current = ""

    if current:
        tokens.append((dictionary.get(current[:-1], 0), current[-1]))

    return tokens


tokens_lz78 = lz78_encode(texto_original)
print(f"{len(tokens_lz78)} tokens")
for idx, ch in tokens_lz78[:15]:
    print(f"  ({idx}, {repr(ch)})")

Serialização dos tokens em bits (16 + 8) e gravação do `.bin`.

In [ ]:
def lz78_tokens_to_bits(tokens):
    return "".join(format(idx, '016b') + format(ord(ch), '08b') for idx, ch in tokens)


bitstream_lz78 = lz78_tokens_to_bits(tokens_lz78)
data_lz78 = bits_to_bytes(bitstream_lz78)  # 24 bits por token → sempre múltiplo de 8

with open("enviar/lz78_codificado.bin", "wb") as f:
    f.write(data_lz78)

print(f"Original: {tamanho_original} bytes")
print(f"LZ78:     {len(data_lz78)} bytes ({len(data_lz78)/tamanho_original:.1%})")

In [ ]:
print(f"texto_original.txt          {tamanho_original:>5} bytes")
print(f"enviar/huffman_codificado.bin {len(data_huffman):>4} bytes  ({len(data_huffman)/tamanho_original:.1%})")
print(f"enviar/lz78_codificado.bin    {len(data_lz78):>4} bytes  ({len(data_lz78)/tamanho_original:.1%})")

## Decodificação

Os arquivos `enviar/huffman_codificado.bin` e `enviar/lz78_codificado.bin` vão para o professor. As respostas com erros são colocadas em `recebido/` (com qualquer nome, `.txt` ou `.bin`). O texto decodificado final é gravado em `decodificado/`.

Converte arquivos recebidos em formato texto para `.bin`, se necessário.

In [ ]:
def txt_para_bin(arquivo_txt, arquivo_bin):
    """Converte arquivo .txt com string '0'/'1' para .bin empacotado em bytes."""
    with open(arquivo_txt, "r") as f:
        bitstream = "".join(c for c in f.read() if c in "01")
    data = bits_to_bytes(bitstream)
    with open(arquivo_bin, "wb") as f:
        f.write(data)
    print(f"{arquivo_txt} ({len(bitstream)} bits) → {arquivo_bin} ({len(data)} bytes)")


# Se o professor mandou de volta como .txt, converte para .bin na mesma pasta
for txt, binario in [
    ("recebido/huffman_com_erros.txt", "recebido/huffman_com_erros.bin"),
    ("recebido/lz78_com_erros.txt", "recebido/lz78_com_erros.bin"),
]:
    if os.path.exists(txt):
        txt_para_bin(txt, binario)
    else:
        print(f"{txt} não encontrado, pulando")

Decodificador Huffman e verificação do arquivo sem erros.

In [ ]:
def huffman_decode(bitstream, codebook):
    inverse = {code: ch for ch, code in codebook.items()}
    decoded = []
    buffer = ""

    for bit in bitstream:
        buffer += bit
        if buffer in inverse:
            decoded.append(inverse[buffer])
            buffer = ""

    if buffer:
        decoded.append(f"[BITS_RESID:{len(buffer)}]")

    return "".join(decoded)


with open("enviar/huffman_codebook.json", "r") as f:
    meta = json.load(f)
codebook_huffman = meta["codebook"]
bit_length_huff = meta["bit_length"]

with open("enviar/huffman_codificado.bin", "rb") as f:
    bits_huff_original = bytes_to_bits(f.read(), bit_length_huff)

texto_huff_sem_erro = huffman_decode(bits_huff_original, codebook_huffman)
print(f"Sem erros: match = {texto_huff_sem_erro == texto_original}")

Decodificação do arquivo com erros. Grava o resultado em `decodificado/`.

In [ ]:
ARQUIVO_HUFFMAN_ERROS = "recebido/huffman_com_erros.bin"

if os.path.exists(ARQUIVO_HUFFMAN_ERROS):
    with open(ARQUIVO_HUFFMAN_ERROS, "rb") as f:
        bits_huff_erros = bytes_to_bits(f.read(), bit_length_huff)

    texto_huff_com_erro = huffman_decode(bits_huff_erros, codebook_huffman)

    with open("decodificado/huffman_decodificado.txt", "w", encoding="utf-8") as f:
        f.write(texto_huff_com_erro)

    bits_diferentes = sum(a != b for a, b in zip(bits_huff_original, bits_huff_erros))
    chars_errados = sum(a != b for a, b in zip(texto_original, texto_huff_com_erro))
    chars_errados += abs(len(texto_original) - len(texto_huff_com_erro))

    print(f"Bits alterados: {bits_diferentes}/{len(bits_huff_original)}")
    print(f"Chars afetados: {chars_errados}/{len(texto_original)}")
    print(f"Tamanho decodificado: {len(texto_huff_com_erro)} (original: {len(texto_original)})")
    print(f"Salvo em decodificado/huffman_decodificado.txt")
    print()
    print(texto_huff_com_erro)
else:
    print(f"{ARQUIVO_HUFFMAN_ERROS} não encontrado")

Decodificador LZ78 e verificação do arquivo sem erros.

In [ ]:
def lz78_bits_to_tokens(bitstream):
    tokens = []
    i = 0
    while i + 24 <= len(bitstream):
        idx = int(bitstream[i:i+16], 2)
        ch = chr(int(bitstream[i+16:i+24], 2))
        tokens.append((idx, ch))
        i += 24

    residual = len(bitstream) - i
    if residual > 0:
        tokens.append((-1, str(residual)))

    return tokens


def lz78_decode(tokens):
    dictionary = {}
    result = []
    next_index = 1

    for idx, ch in tokens:
        if idx == -1:
            result.append(f"[BITS_RESID:{ch}]")
            continue

        if idx == 0:
            entry = ch
        elif idx in dictionary:
            entry = dictionary[idx] + ch
        else:
            entry = f"[ERR:{idx}]" + ch

        result.append(entry)
        if next_index <= 65535:
            dictionary[next_index] = entry
            next_index += 1

    return "".join(result)


with open("enviar/lz78_codificado.bin", "rb") as f:
    bits_lz78_original = bytes_to_bits(f.read())

texto_lz78_sem_erro = lz78_decode(lz78_bits_to_tokens(bits_lz78_original))
print(f"Sem erros: match = {texto_lz78_sem_erro == texto_original}")

Decodificação do arquivo com erros. Grava o resultado em `decodificado/`.

In [ ]:
ARQUIVO_LZ78_ERROS = "recebido/lz78_com_erros.bin"

if os.path.exists(ARQUIVO_LZ78_ERROS):
    with open(ARQUIVO_LZ78_ERROS, "rb") as f:
        bits_lz78_erros = bytes_to_bits(f.read())

    texto_lz78_com_erro = lz78_decode(lz78_bits_to_tokens(bits_lz78_erros))

    with open("decodificado/lz78_decodificado.txt", "w", encoding="utf-8") as f:
        f.write(texto_lz78_com_erro)

    bits_diferentes = sum(a != b for a, b in zip(bits_lz78_original, bits_lz78_erros))
    chars_errados = sum(a != b for a, b in zip(texto_original, texto_lz78_com_erro))
    chars_errados += abs(len(texto_original) - len(texto_lz78_com_erro))

    print(f"Bits alterados: {bits_diferentes}/{len(bits_lz78_original)}")
    print(f"Chars afetados: {chars_errados}/{len(texto_original)}")
    print(f"Tamanho decodificado: {len(texto_lz78_com_erro)} (original: {len(texto_original)})")
    print(f"Salvo em decodificado/lz78_decodificado.txt")
    print()
    print(texto_lz78_com_erro)
else:
    print(f"{ARQUIVO_LZ78_ERROS} não encontrado")

## Comparação

Diferença bit-a-bit entre original e corrompido, e impacto no texto reconstruído.

In [ ]:
arquivos = [
    ("Huffman", "recebido/huffman_com_erros.bin", "enviar/huffman_codificado.bin", bit_length_huff,
     lambda b: huffman_decode(b, codebook_huffman)),
    ("LZ78", "recebido/lz78_com_erros.bin", "enviar/lz78_codificado.bin", None,
     lambda b: lz78_decode(lz78_bits_to_tokens(b))),
]

for nome, arq_erros, arq_orig, bit_len, decode_fn in arquivos:
    if not os.path.exists(arq_erros):
        print(f"{nome}: {arq_erros} não encontrado")
        continue

    with open(arq_orig, "rb") as f:
        bits_orig = bytes_to_bits(f.read(), bit_len)
    with open(arq_erros, "rb") as f:
        bits_erros = bytes_to_bits(f.read(), bit_len)

    bits_diff = sum(a != b for a, b in zip(bits_orig, bits_erros))
    texto_dec = decode_fn(bits_erros)

    chars_diff = sum(a != b for a, b in zip(texto_original, texto_dec))
    chars_diff += abs(len(texto_original) - len(texto_dec))

    print(nome)
    print(f"  bits flipados: {bits_diff}/{len(bits_orig)}")
    print(f"  chars errados: {chars_diff}/{len(texto_original)}")
    if bits_diff > 0:
        print(f"  propagação: {chars_diff/bits_diff:.1f} chars/bit")
    print()

## Notas

Huffman, por ser de comprimento variável, dessincroniza com qualquer bit errado — um único flip pode corromper todo o resto do texto e mudar o comprimento total. LZ78, com tokens de 24 bits, contém o erro no token onde aconteceu; se o bit cair no índice, pode apontar para outra entrada e propagar parcialmente (ou ser inválido, daí `[ERR:idx]`). Mais resiliente, não imune.